[Back to NLP guideline](Natural-Language-Processing.html)


## **NLP Tasks and Applications** {#nlp-tasks-and-applications}

An NLP task turns an open-ended language problem into a learnable contract. The contract states what information enters the system, what output must be produced, what supervision is available, and how success will be judged. "Understand this review" is not yet a task. "Given a review, predict one of `positive`, `neutral`, or `negative`" is a task because its input, output space, and evaluation target are explicit.

The same real application often contains several tasks. A customer-support assistant may classify intent, extract product names, retrieve a policy, generate a response, and estimate whether the user is becoming frustrated. Treating the whole product as one opaque generation problem hides these different error modes. Task decomposition makes data collection, model selection, evaluation, and human review much easier to reason about.

This chapter groups tasks by the structure of the information they produce:

| Task Family | Typical Input | Typical Output | Central Question |
|---|---|---|---|
| understanding and prediction | one text, a text pair, or a token sequence | label, relation, or aligned labels | what does the text express or imply? |
| knowledge access | query plus a collection or passage | ranked evidence or answer | where is the relevant information? |
| generation | source text, dialogue context, or instruction | a new text sequence | what text should be produced? |
| affective computing | language plus optional speaker/context signals | sentiment, emotion, appraisal, or empathetic action | what affect is expressed, experienced, or appropriate? |

The groups are not rigid boundaries. Extractive question answering predicts token spans, while generative question answering produces a sequence. Information retrieval may be a complete search task or an internal component of a retrieval-augmented generator. The useful habit is to identify the actual input-output contract before choosing the model.

### **From Language Problems to Task Formulations** {#from-language-problems-to-task-formulations}

#### **Task Families, Inputs, Outputs, and Supervision** {#task-families-inputs-outputs-and-supervision}

A task formulation can be described by a dataset of examples

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N},
$$

where $N$ is the number of examples, $x_i$ is the input representation for example $i$, and $y_i$ is its desired output. This notation looks simple, but the meaning of $x_i$ and $y_i$ changes the entire learning problem.

- In document classification, $x_i$ is a document and $y_i$ is one label.
- In named entity recognition, $x_i = (x_{i1}, \ldots, x_{iT})$ is a token sequence and $y_i = (y_{i1}, \ldots, y_{iT})$ is an aligned tag sequence.
- In natural language inference, $x_i$ is a pair `(premise, hypothesis)` and $y_i$ is their semantic relation.
- In translation, $x_i$ is a source-language sequence and $y_i$ is a target-language sequence whose length may differ.
- In retrieval, one training item may contain a query, a relevant document, and one or more non-relevant documents.

Supervision is equally important. A gold label supplied by an expert, a click collected from users, a translation aligned by a publisher, and a response generated by another model do not have the same reliability or meaning. Before training, ask what process produced the target and whether the target truly represents the intended construct.

Modern models can present many tasks through one text-to-text interface. A prefix such as `translate English to German:` or `summarize:` tells the model what output to generate. This unifies the software interface, but it does not erase task differences: classification still needs label calibration, translation still needs adequacy, and summarization still needs factuality checks.

> ![Different NLP tasks represented through a unified text-to-text interface](assets/nlp-task-text-to-text.png){width=92%}
>
> The figure shows translation, acceptability classification, semantic similarity, and summarization sharing one text-to-text model interface. Source: [Raffel et al., Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://www.jmlr.org/papers/v21/20-074.html).

#### **The Task Contract** {#the-task-contract}

A robust task contract records more than a task name. It should state:

| Contract Element | Question | Example |
|---|---|---|
| unit of prediction | what receives one prediction? | one support message |
| input context | what evidence is available? | message plus previous two turns |
| output schema | what values are legal? | `{intent, confidence, entities}` |
| label semantics | what does each target mean? | `refund_request` means an explicit request for money back |
| supervision source | who or what created the target? | two trained annotators with adjudication |
| abstention policy | when may the system decline? | confidence below a calibrated threshold |
| main metric | what behavior is optimized? | macro F1 across intents |
| critical slices | where must quality be checked separately? | short messages, code-switching, new products |
| downstream action | what happens after prediction? | route to billing queue |
| error cost | which mistakes matter most? | missing an account-compromise report |

This contract prevents a common failure: training an accurate model for the wrong operational question. For example, predicting the topic of a message is not the same as predicting which team should handle it. Topic and routing may correlate in historical data, but routing also depends on policy, urgency, and available staff.

<details>
<summary>Python Representing and Validating a Task Contract</summary>

```python
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class TaskContract:
    name: str
    input_fields: tuple[str, ...]
    output_labels: tuple[str, ...]
    main_metric: str
    allow_abstention: bool = True

    def validate_example(self, example: dict) -> None:
        # Step 1: verify that every required input is present.
        missing = [field for field in self.input_fields if field not in example]
        if missing:
            raise ValueError(f"missing input fields: {missing}")

        # Step 2: verify that supervised targets follow the declared schema.
        if "label" in example and example["label"] not in self.output_labels:
            raise ValueError(f"unknown label: {example['label']}")

contract = TaskContract(
    name="support_intent",
    input_fields=("message", "previous_turn"),
    output_labels=("refund", "delivery", "account", "other"),
    main_metric="macro_f1",
)

training_example = {
    "message": "The reset link has expired again.",
    "previous_turn": "I cannot sign in.",
    "label": "account",
}

contract.validate_example(training_example)
print("example matches the task contract")
```

</details>

The task contract is also the bridge to evaluation. The metric must reflect the output structure, and the test set must reflect the intended context. A token classifier cannot be evaluated only with document accuracy, and a dialogue agent cannot be evaluated only with the fluency of one response.

### **Understanding and Prediction Tasks** {#understanding-and-prediction-tasks}

Understanding tasks map language to a constrained output such as a category, a relation, or a label attached to each token. Their outputs are easier to validate than open-ended text, but their apparent simplicity can hide difficult decisions about label meaning, context, and class imbalance.

#### **Text Classification** {#text-classification}

Text classification assigns one or more labels to a span of text. The span may be a sentence, review, email, document, or complete conversation. Common applications include topic routing, spam detection, intent recognition, toxicity detection, sentiment analysis, and document coding.

For single-label classification with $K$ classes, the model estimates

$$
p(y=k \mid x), \qquad k \in \{1, \ldots, K\}.
$$

$x$ is the input text, $y$ is the class variable, and $p(y=k \mid x)$ is the model's estimated probability that class $k$ is correct. The predicted class is often

$$
\hat{y} = \arg\max_k p(y=k \mid x).
$$

The `argmax` selects the class with the largest probability. It does not tell us whether that probability is calibrated or whether the model should be trusted on this input. In a high-impact workflow, a threshold or abstention rule may be more appropriate than always selecting a label.

Three output settings should be separated:

| Setting | Output Rule | Example |
|---|---|---|
| binary | one of two labels | spam vs not spam |
| multiclass | exactly one of $K$ labels | billing, delivery, account, other |
| multilabel | any subset of labels | a complaint can be both `delivery` and `refund` |

Classification quality depends heavily on the unit of prediction. A whole review can be labeled negative even though it praises the product and criticizes only delivery. If the application needs this distinction, aspect-based sentiment or span extraction is a better formulation than document-level classification.

<details>
<summary>Python Training a Reproducible TF-IDF Classification Baseline</summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

train_texts = [
    "My parcel still has not arrived",
    "Please refund the duplicate charge",
    "The password reset link has expired",
    "Where can I download the invoice?",
]
train_labels = ["delivery", "refund", "account", "billing"]

test_texts = [
    "The tracking page has not changed for a week",
    "I was charged twice and need my money back",
]
test_labels = ["delivery", "refund"]

# Step 1: learn lexical features only from the training split.
# Keeping vectorization inside the pipeline prevents test-data leakage.
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=7)),
])

# Step 2: fit the complete baseline under one reproducible configuration.
model.fit(train_texts, train_labels)

# Step 3: inspect predictions and class probabilities, not only accuracy.
predictions = model.predict(test_texts)
probabilities = model.predict_proba(test_texts)

for text, label, scores in zip(test_texts, predictions, probabilities):
    confidence = max(scores)
    print(f"{label:>8}  confidence={confidence:.3f}  text={text}")

print(classification_report(test_labels, predictions, zero_division=0))
```

</details>

This lexical baseline is valuable even when the final system uses a Transformer. If a complex model does not improve substantially over TF-IDF, the dataset may reward keywords rather than contextual reasoning. The baseline chapter explains how such comparisons should be controlled, while the evaluation chapter explains macro F1, calibration, and slice analysis in detail.

#### **Sequence Labeling** {#sequence-labeling}

Sequence labeling predicts an aligned output for each input position. Given tokens

$$
x = (x_1, x_2, \ldots, x_T),
$$

the model produces labels

$$
y = (y_1, y_2, \ldots, y_T).
$$

$T$ is the number of tokens, $x_t$ is the token at position $t$, and $y_t$ is its label. The alignment is the defining property: output position $t$ describes input position $t$. Common tasks include part-of-speech tagging, named entity recognition, chunking, slot filling, and error detection.

A token classifier can predict each $y_t$ independently from a contextual representation. A structured model such as a conditional random field additionally scores transitions between labels. This is useful when valid outputs have dependencies, such as `I-PER` normally following `B-PER` or `I-PER` rather than appearing after `O`.

##### **Part-of-Speech Tagging** {#part-of-speech-tagging}

Part-of-speech tagging assigns a grammatical category such as noun, verb, adjective, or determiner to each token. The label depends on context: `book` is a noun in `read the book` and a verb in `book a ticket`. POS tags support parsing, information extraction, linguistic analysis, and rule-based post-processing, although modern contextual models may learn much of this information internally.

Tagsets represent a theoretical and practical choice. A small universal tagset supports cross-language comparison, while a detailed language-specific tagset captures distinctions that matter for a particular grammar. Tokenization and tagging must agree: contractions, punctuation, multiword expressions, and subword pieces require explicit alignment rules.

##### **Named Entity Recognition** {#named-entity-recognition}

Named entity recognition identifies spans that mention entities and assigns types such as person, organization, location, product, or date. It is usually represented with BIO tags:

```text
Wei       B-PER
joined    O
Sydney    B-LOC
Data      B-ORG
Lab       I-ORG
```

`B-X` begins an entity of type `X`, `I-X` continues it, and `O` marks tokens outside entities. NER is not simply proper-noun detection. Domain entities may be lowercase (`iPhone`, gene names), common words may become product names, and the required ontology changes across medicine, law, finance, and customer support.

<details>
<summary>Python Converting BIO Tags into Entity Spans</summary>

```python
tokens = ["Wei", "joined", "Sydney", "Data", "Lab", "today"]
tags = ["B-PER", "O", "B-LOC", "B-ORG", "I-ORG", "O"]

def bio_to_spans(tokens, tags):
    spans = []
    current_tokens = []
    current_type = None
    start = None

    # Step 1: append a sentinel O tag so the final entity is closed.
    for index, (token, tag) in enumerate(zip(tokens + [None], tags + ["O"])):
        prefix, entity_type = (tag.split("-", 1) + [None])[:2]

        # Step 2: close the current entity when a new span or O begins.
        if current_tokens and (prefix in {"B", "O"} or entity_type != current_type):
            spans.append({
                "text": " ".join(current_tokens),
                "type": current_type,
                "token_start": start,
                "token_end": index,  # exclusive
            })
            current_tokens, current_type, start = [], None, None

        # Step 3: start or continue a valid entity span.
        if prefix == "B":
            current_tokens = [token]
            current_type = entity_type
            start = index
        elif prefix == "I" and current_type == entity_type:
            current_tokens.append(token)

    return spans

for entity in bio_to_spans(tokens, tags):
    print(entity)
```

</details>

Sequence labeling should normally be evaluated at the structure that users care about. For NER, an entity is correct only when both its boundary and type are correct. Token accuracy can look excellent because most tokens are `O` while the system still misses nearly every entity.

#### **Natural Language Inference** {#natural-language-inference}

Natural language inference (NLI) predicts whether a hypothesis follows from a premise. The standard labels are:

| Label | Meaning | Example |
|---|---|---|
| entailment | the premise provides sufficient support | `A child is running` -> `Someone is moving` |
| contradiction | the hypothesis conflicts with the premise | `A child is running` -> `Nobody is moving` |
| neutral | the premise neither proves nor disproves it | `A child is running` -> `The child is late for school` |

The model estimates $p(y \mid p, h)$, where $p$ is the premise, $h$ is the hypothesis, and $y$ is the relation label. NLI requires lexical knowledge, negation, quantifiers, coreference, world knowledge, and sometimes pragmatic assumptions. It is therefore widely used as a diagnostic task for sentence-pair reasoning.

The labels are directional. If `A poodle is a dog` entails `A dog is present`, the reverse does not entail that a poodle is present. Dataset artifacts can also make the task easier than intended. For example, annotators may use characteristic words such as `nobody` when writing contradictions, allowing a model to exploit the hypothesis without reading the premise.

<details>
<summary>Python Checking Direction and Consistency in NLI Predictions</summary>

```python
examples = [
    {
        "premise": "Every approved request received a confirmation email.",
        "hypothesis": "Some approved requests received confirmation emails.",
        "label": "entailment",
    },
    {
        "premise": "No account was suspended.",
        "hypothesis": "At least one account was suspended.",
        "label": "contradiction",
    },
]

def build_pair(premise, hypothesis):
    # Step 1: preserve field names so their direction cannot be swapped silently.
    return f"premise: {premise} hypothesis: {hypothesis}"

for item in examples:
    forward = build_pair(item["premise"], item["hypothesis"])
    reverse = build_pair(item["hypothesis"], item["premise"])

    # Step 2: create a directional contrast case for evaluation.
    print("gold:", item["label"])
    print("forward:", forward)
    print("reverse:", reverse)
    print()
```

</details>

In applications, NLI can support fact verification, consistency checking, zero-shot classification, and detecting whether a generated answer is supported by evidence. However, an NLI score is not a universal truth detector. The result is conditioned on the supplied premise and the relation definitions learned from the training data.

| Understanding Task | Output Granularity | Best Fit | Main Risk |
|---|---|---|---|
| text classification | sentence or document | routing, topic, intent | labels hide local evidence |
| sequence labeling | token or span | NER, slots, linguistic tags | boundary and tokenization mismatch |
| NLI | relationship between texts | entailment and consistency | annotation artifacts and missing world context |

### **Knowledge Access Tasks** {#knowledge-access-tasks}

Knowledge access tasks connect a language request to information stored outside the immediate input. Retrieval returns evidence candidates; question answering transforms evidence into a direct answer. In modern systems these tasks are often chained, but they should still be evaluated separately because a generator cannot recover evidence that retrieval never supplied.

#### **Information Retrieval and Ranking** {#information-retrieval-and-ranking}

Information retrieval (IR) receives a query $q$ and a collection of documents $\mathcal{C}$, then ranks documents by estimated relevance. The result is an ordered list rather than one class label:

$$
d_{(1)}, d_{(2)}, \ldots, d_{(k)},
$$

where $d_{(1)}$ is the highest-ranked document and $k$ is the number returned. Search engines, legal discovery, literature search, support knowledge bases, and RAG systems all depend on this operation.

Lexical retrieval such as BM25 rewards exact term overlap while correcting for document length and common words. Dense retrieval encodes queries and documents as vectors and uses similarity, often a dot product or cosine similarity. Lexical retrieval is strong for names, identifiers, and rare terms; dense retrieval is strong when relevant text uses different wording. Hybrid retrieval combines both signals:

$$
s_{\text{hybrid}}(q,d) = \alpha s_{\text{lexical}}(q,d)
+ (1-\alpha)s_{\text{dense}}(q,d).
$$

$s_{\text{lexical}}$ and $s_{\text{dense}}$ are normalized scores, and $\alpha \in [0,1]$ controls their balance. The formula is meaningful only if the scores are on comparable scales. Rank fusion methods avoid this calibration problem by combining rank positions rather than raw scores.

Retrieval supervision may come from expert relevance judgments, clicked results, purchases, citations, or question-answer pairs. Behavioral signals are plentiful but biased by the existing ranking: users cannot click a relevant result they never saw.

<details>
<summary>Python Combining Lexical and Dense Rankings with Reciprocal Rank Fusion</summary>

```python
lexical_ranking = ["doc_policy_7", "doc_refund_2", "doc_account_4"]
dense_ranking = ["doc_refund_2", "doc_policy_7", "doc_delivery_9"]

def reciprocal_rank_fusion(rankings, k=60):
    scores = {}

    # Step 1: convert each rank into a bounded contribution.
    # The constant k prevents the top position from dominating too strongly.
    for ranking in rankings:
        for rank, document_id in enumerate(ranking, start=1):
            scores[document_id] = scores.get(document_id, 0.0) + 1 / (k + rank)

    # Step 2: sort the merged candidates by their accumulated rank evidence.
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)

for document_id, score in reciprocal_rank_fusion(
    [lexical_ranking, dense_ranking]
):
    print(f"{score:.5f}  {document_id}")
```

</details>

The first retrieval question is usually recall: did the candidate set contain the needed evidence? A reranker can improve ordering but cannot rescue a missing document. Retrieval quality also depends on document segmentation, metadata filters, freshness, duplicates, and access permissions, which are developed further in the modern systems chapter.

#### **Question Answering** {#question-answering}

Question answering (QA) produces an answer to a natural-language question. The task varies according to where the evidence comes from and how the answer is represented.

| QA Setting | Evidence | Output |
|---|---|---|
| extractive | supplied passage | a contiguous text span |
| multiple choice | question plus candidates | one candidate label |
| open-domain | large external collection | retrieved evidence plus answer |
| generative | passage, retrieved documents, or model knowledge | free-form answer sequence |
| conversational | current question plus dialogue history | context-dependent answer |

In extractive QA, an encoder produces a representation for every context token. Two prediction heads estimate start and end probabilities:

$$
P(i,j \mid q,c) = P_{\text{start}}(i \mid q,c)
P_{\text{end}}(j \mid q,c), \qquad i \leq j.
$$

$q$ is the question, $c$ is the context, $i$ is the answer's start position, and $j$ is its end position. The model selects a valid span with a high joint score. The product assumes the start and end heads provide compatible evidence; implementations also restrict maximum answer length and compare the best span with a no-answer score.

Generative QA can synthesize information and answer in natural language, but it can also introduce claims absent from the evidence. A production QA system should preserve evidence provenance and distinguish `the documents do not answer this` from `the model can produce a plausible sentence`.

<details>
<summary>Python Selecting a Valid Extractive QA Span</summary>

```python
tokens = ["Refunds", "are", "processed", "within", "five", "business", "days", "."]
start_prob = [0.01, 0.01, 0.03, 0.12, 0.70, 0.08, 0.03, 0.02]
end_prob =   [0.01, 0.01, 0.02, 0.03, 0.05, 0.20, 0.65, 0.03]

def best_span(tokens, start_prob, end_prob, max_answer_tokens=5):
    best = {"score": -1.0, "start": None, "end": None}

    # Step 1: enumerate only legal spans with start <= end.
    for start in range(len(tokens)):
        last_end = min(len(tokens), start + max_answer_tokens)
        for end in range(start, last_end):
            score = start_prob[start] * end_prob[end]

            # Step 2: retain the highest-scoring valid answer span.
            if score > best["score"]:
                best = {"score": score, "start": start, "end": end}

    # Step 3: convert inclusive token boundaries back into answer text.
    answer = " ".join(tokens[best["start"]:best["end"] + 1])
    return answer, best

answer, evidence = best_span(tokens, start_prob, end_prob)
print("answer:", answer)
print("span evidence:", evidence)
```

</details>

Retrieval and QA solve different parts of knowledge access. Retrieval asks which evidence should be read; QA asks what answer follows from that evidence. Measuring only answer quality can conceal weak retrieval, while measuring only retrieval can conceal unsupported generation.

| System | Retrieval Needed? | Answer Construction | Appropriate When |
|---|---:|---|---|
| document search | yes | user reads results | exploration and high user control |
| extractive QA | sometimes | copy a span | answers appear explicitly in text |
| generative QA | often | synthesize a response | evidence must be combined or explained |
| RAG assistant | yes | retrieve, generate, and cite | knowledge changes and provenance matters |

### **Generation Tasks** {#generation-tasks}

Generation tasks produce a variable-length sequence. A common formulation models the probability of an output $y=(y_1,\ldots,y_M)$ conditioned on input $x$:

$$
p(y \mid x) = \prod_{t=1}^{M} p(y_t \mid y_{<t}, x).
$$

$M$ is the output length, $y_t$ is the token generated at step $t$, and $y_{<t}$ contains all previously generated tokens. The product expresses autoregressive generation: every next-token decision depends on the source and the partial output. A locally likely token can lead to a poor full sequence, which is why decoding strategy matters.

Fluent text is not automatically correct text. Each generation task imposes additional constraints such as meaning preservation, evidence faithfulness, dialogue consistency, style, safety, or terminology.

#### **Machine Translation** {#machine-translation}

Machine translation (MT) converts a source-language sequence into a target-language sequence while preserving meaning and producing natural target-language text. Translation is not word substitution. Word order, morphology, idioms, omitted subjects, politeness, gender, and cultural conventions may require substantial restructuring.

Neural MT commonly uses an encoder-decoder model. The encoder represents the source; the decoder generates target tokens while attending to relevant source positions. Multilingual models share parameters across many language pairs and can transfer useful patterns to lower-resource languages, but shared capacity and imbalanced data can favor high-resource languages.

Training normally uses parallel sentences $(x,y)$ and minimizes token-level negative log-likelihood. At inference, greedy or beam decoding produces a candidate translation. The model may still omit content, mistranslate names, over-literalize idioms, or hallucinate when the source is unusual. Terminology constraints and human review remain important in legal, medical, and technical domains.

<details>
<summary>Python Running Translation with an Explicit Model and Quality Checks</summary>

```python
from transformers import pipeline

# Step 1: pin a concrete model instead of relying on an unspecified default.
translator = pipeline(
    task="translation",
    model="Helsinki-NLP/opus-mt-en-de",
)

source = "The refund will be issued within five business days."

# Step 2: generate a bounded translation.
result = translator(source, max_new_tokens=80)[0]["translation_text"]
print("source:", source)
print("translation:", result)

# Step 3: preserve terms that require separate verification.
# Real systems use terminology lists, named-entity checks, and back-translation
# or bilingual human review for high-impact content.
required_concepts = ["refund", "five", "business days"]
print("concepts requiring QA:", required_concepts)
```

</details>

Automatic metrics such as BLEU, chrF, and learned semantic metrics are useful for comparing systems over datasets, but they do not certify one translation. Human evaluation should consider adequacy, fluency, terminology, and consequences of an error.

#### **Text Summarization** {#text-summarization}

Text summarization compresses a source while preserving its most important information. **Extractive** summarization selects existing sentences or spans. **Abstractive** summarization generates new wording and can combine information, but it introduces a greater risk of unsupported details.

The target depends on purpose. A news headline, clinical handover, meeting action list, and legal case brief require different content selection. Compression can be described by

$$
\text{compression ratio} = \frac{|y|}{|x|},
$$

where $|x|$ is source length and $|y|$ is summary length in the same unit, such as tokens. A smaller ratio means stronger compression, but it does not imply better summarization. Excessive compression can remove qualifications, uncertainty, or minority viewpoints.

Attention-based encoder-decoder models learn which source positions are relevant while generating each summary token. The diagram below illustrates this alignment: the decoder uses a context vector derived from attention over source hidden states.

> ![Attention-based sequence-to-sequence summarization](assets/summarization-attention-seq2seq.png){width=94%}
>
> The source words influence the context vector used to generate the next summary word. Source: [See, Liu, and Manning, Get To The Point: Summarization with Pointer-Generator Networks](https://aclanthology.org/P17-1099/).

Modern Transformer summarizers replace recurrent states with self-attention and cross-attention, but the source-to-summary alignment problem remains. A summary can be fluent yet distort who did what, change a number, or turn uncertainty into certainty.

<details>
<summary>Python Building an Inspectable Extractive Summarization Baseline</summary>

```python
import re
from collections import Counter

document = (
    "The support team introduced a new refund workflow. "
    "Requests are now checked automatically for duplicate charges. "
    "High-value refunds still require manual approval. "
    "The change reduced average handling time by 18 percent."
)

def extractive_summary(text, sentence_count=2):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    words = re.findall(r"[a-z]+", text.lower())
    frequencies = Counter(words)

    scored = []
    for position, sentence in enumerate(sentences):
        sentence_words = re.findall(r"[a-z]+", sentence.lower())

        # Step 1: score a sentence by normalized content-word frequency.
        score = sum(frequencies[word] for word in sentence_words)
        score /= max(len(sentence_words), 1)
        scored.append((score, position, sentence))

    # Step 2: choose strong sentences, then restore source order.
    selected = sorted(scored, reverse=True)[:sentence_count]
    selected = sorted(selected, key=lambda item: item[1])
    return " ".join(sentence for _, _, sentence in selected)

summary = extractive_summary(document)
print(summary)

# Step 3: compare source and summary lengths as a diagnostic.
ratio = len(summary.split()) / len(document.split())
print(f"compression ratio: {ratio:.2f}")
```

</details>

An extractive baseline is less flexible but provides a useful factuality reference because every selected sentence comes from the source. A strong abstractive system should be evaluated for coverage, relevance, coherence, and claim-level faithfulness rather than ROUGE alone.

#### **Dialogue and Conversational Systems** {#dialogue-and-conversational-systems}

A dialogue system chooses responses or actions over multiple turns. Unlike isolated generation, the meaning of the current input depends on conversation state:

```text
User: I need to change it.
```

Without previous turns, neither `it` nor the intended change is known. Dialogue systems therefore maintain some representation of history, entities, user goals, tool results, and unresolved questions.

Task-oriented dialogue aims to complete a goal such as booking, troubleshooting, or account management. Open-domain dialogue prioritizes engaging and contextually appropriate conversation. Many deployed assistants combine both: free-form language surrounds a constrained workflow.

A classical task-oriented pipeline contains:

```text
user utterance
-> intent and slot extraction
-> dialogue state update
-> policy or action selection
-> tool execution
-> response generation
```

End-to-end language models may perform several steps in one model call, but the system still needs explicit state and validation when actions have real consequences.

<details>
<summary>Python Maintaining Explicit Dialogue State Before Taking an Action</summary>

```python
from dataclasses import dataclass

@dataclass
class RefundState:
    order_id: str | None = None
    reason: str | None = None
    confirmed: bool = False

def next_action(state: RefundState) -> str:
    # Step 1: request missing information rather than inventing it.
    if state.order_id is None:
        return "ask_for_order_id"
    if state.reason is None:
        return "ask_for_reason"

    # Step 2: require explicit confirmation before a consequential tool call.
    if not state.confirmed:
        return "ask_for_confirmation"

    # Step 3: only a complete, confirmed state may reach the refund tool.
    return "submit_refund_request"

state = RefundState(order_id="A-1042", reason="duplicate charge")
print(next_action(state))

state.confirmed = True
print(next_action(state))
```

</details>

Dialogue quality is multidimensional. A response can be fluent but fail the task, complete the task while being impolite, or follow the immediate turn while contradicting earlier information. Evaluation therefore combines task success, response quality, state accuracy, safety, and user outcomes.

| Generation Task | Meaning Constraint | Typical Failure | Useful Control |
|---|---|---|---|
| translation | preserve source meaning | omission or mistranslation | terminology checks and bilingual review |
| summarization | preserve important supported claims | factual distortion | source attribution and claim verification |
| dialogue | remain consistent with state and goal | invented state or unsafe action | explicit state, tools, confirmation, and fallback |

### **Affective Computing and Sentiment Analysis** {#affective-computing-and-sentiment-analysis}

Affective computing studies systems that recognize, represent, or respond to affective phenomena such as sentiment, emotion, mood, appraisal, stance, and empathy. Text provides indirect evidence of affect rather than a direct measurement of a person's internal state. The sentence `Fine, do whatever you want` may express resignation, anger, permission, or something else depending on speaker, history, and tone.

This distinction is foundational:

| Target | Question | Example Output |
|---|---|---|
| sentiment | is the evaluation positive or negative? | negative |
| emotion expression | what emotion is expressed in the text? | frustration |
| experienced emotion | what may the speaker feel? | uncertain, possibly frustration |
| stance | what position is taken toward a target? | against policy change |
| empathy | what response acknowledges and supports the user? | validation plus appropriate help |

A model should not silently move from expression to diagnosis. Detecting negative language does not establish a person's mental health condition, intent, or stable personality.

#### **Representing Affect** {#representing-affect}

Affect can be represented categorically or dimensionally. Categorical schemes use labels such as joy, anger, sadness, fear, or finer-grained emotions. Dimensional schemes place affect in a continuous space, commonly including **valence** (pleasant to unpleasant) and **arousal** (calm to activated). A third dimension such as dominance is sometimes added.

Neither representation is universally correct. Categories are interpretable for routing and analysis; dimensions represent intensity and similarity more smoothly. Fine-grained categories create sparse labels and disagreement, while coarse categories can erase important distinctions such as disappointment versus anger.

The GoEmotions data illustrates both label imbalance and relationships among emotions. Some labels are much more frequent, and related emotions form correlated clusters. This is a reminder that emotion labels are not independent, equally common boxes.

> ![Frequency and correlation structure of fine-grained emotion labels](assets/goemotions-label-structure.png){width=95%}
>
> The left panel shows label frequency and annotator agreement; the right panel shows correlations and hierarchical grouping among emotions. Source: [Demszky et al., GoEmotions: A Dataset of Fine-Grained Emotions](https://aclanthology.org/2020.acl-main.372/).

#### **Sentiment Analysis** {#sentiment-analysis}

Sentiment analysis predicts evaluative polarity, often positive, neutral, or negative. Document-level sentiment assumes one dominant judgment. Sentence-level sentiment localizes it. **Aspect-based sentiment analysis** identifies both the target aspect and the sentiment toward it:

```text
The camera is excellent, but the battery is disappointing.

camera  -> positive
battery -> negative
```

The aspect formulation is more actionable for product analysis because it separates mixed opinions. It is also harder: the system must identify aspect spans, connect opinions to the correct aspect, handle implicit targets, and interpret negation and comparison.

Sentiment is domain dependent. `unpredictable` may be negative for a car but positive for a thriller. Star ratings are convenient distant supervision, but a three-star review may contain both praise and severe criticism. Sarcasm, politeness, quoted speech, and code-switching further complicate the mapping.

#### **Emotion Recognition** {#emotion-recognition}

Emotion recognition often uses multilabel classification because one utterance can express several emotions. For each emotion $k$, the model predicts an independent sigmoid probability

$$
p_k = \sigma(z_k) = \frac{1}{1+e^{-z_k}}.
$$

$z_k$ is the model's logit for emotion $k$, $e$ is the base of the natural logarithm, and $\sigma$ maps any real value to $(0,1)$. Unlike softmax, sigmoid probabilities do not have to sum to one, so both `sadness` and `disappointment` can be active.

The multilabel binary cross-entropy loss is

$$
\mathcal{L} = -\frac{1}{K}\sum_{k=1}^{K}
\left[y_k\log p_k + (1-y_k)\log(1-p_k)\right].
$$

$K$ is the number of emotion labels, $y_k \in \{0,1\}$ is the gold indicator, and $p_k$ is the predicted probability. The first term penalizes low probability for a present emotion; the second penalizes high probability for an absent emotion. Rare emotions may require class weighting, sampling, or label-specific thresholds.

<details>
<summary>Python Applying Label-Specific Thresholds to Emotion Predictions</summary>

```python
probabilities = {
    "anger": 0.44,
    "disappointment": 0.72,
    "sadness": 0.58,
    "neutral": 0.08,
}

# Thresholds should be selected on validation data for the intended metric.
# A rarer or high-impact emotion may need a different operating point.
thresholds = {
    "anger": 0.40,
    "disappointment": 0.55,
    "sadness": 0.60,
    "neutral": 0.50,
}

# Step 1: retain every label that passes its own calibrated threshold.
predicted = [
    label
    for label, probability in probabilities.items()
    if probability >= thresholds[label]
]

# Step 2: preserve scores for review instead of exposing only hard labels.
ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)

print("predicted emotions:", predicted)
print("ranked evidence:", ranked)
```

</details>

Annotator disagreement is not always annotation noise. Readers can legitimately infer different emotions because context is missing or because affective interpretation is subjective. Preserving label distributions or multiple annotations can be more informative than forcing one majority label.

#### **Empathy and Emotion-Aware Interaction** {#empathy-and-emotion-aware-interaction}

Empathy in dialogue is not equivalent to predicting an emotion and inserting a sympathetic phrase. A useful response may need to recognize the situation, validate the concern without making unsupported assumptions, and take an appropriate action.

Consider:

```text
User: I have explained this three times and the charge is still there.
```

A weak response might say `You sound angry.` This labels the user and may escalate the exchange. A stronger response acknowledges the experience and advances the task: `I am sorry you have had to repeat this. I will check the duplicate charge using the case details already provided.`

Emotion-aware systems should therefore separate at least three decisions:

```text
affective evidence in the message
-> uncertainty-aware interpretation
-> response strategy appropriate to context and policy
```

The final strategy may be apology, clarification, de-escalation, celebration, referral, or simply efficient problem solving. In sensitive settings, affect predictions should support trained humans rather than diagnose users or trigger irreversible actions.

| Affect Task | Output | Strong Use Case | Important Limitation |
|---|---|---|---|
| document sentiment | polarity label | aggregate feedback trends | hides aspects and mixed opinions |
| aspect sentiment | aspect plus polarity | product or service diagnosis | implicit aspects are difficult |
| emotion classification | one or more emotion labels | conversation analysis | interpretation is subjective and contextual |
| dimensional affect | valence/arousal scores | tracking gradual change | scales may be hard to explain operationally |
| empathetic response | response strategy or text | support and wellbeing interfaces | apparent empathy can overclaim understanding |

### **Choosing and Scoping an NLP Task** {#choosing-and-scoping-an-nlp-task}

The best task is not necessarily the one supported by the most fashionable model. It is the formulation whose output directly supports a legitimate user or organizational decision and whose errors can be measured and managed.

Start from the desired action:

```text
user or operational need
-> decision to support
-> information required for that decision
-> task output
-> supervision and data
-> model family
-> evaluation and fallback
```

If the goal is to route a support message, multiclass classification may be sufficient. If the goal is to display exactly which product and order are involved, add entity or slot extraction. If the answer depends on policy documents, retrieval is required. If the system must explain the answer naturally, generation may be added after evidence retrieval. Each added component creates value and new failure modes.

#### **Task Selection Matrix** {#task-selection-matrix}

| Need | Recommended Formulation | Output | First Baseline | Main Evaluation Unit |
|---|---|---|---|---|
| assign one case to a queue | multiclass classification | one label | TF-IDF plus logistic regression | case-level macro F1 |
| detect several topics in one document | multilabel classification | label set | one-vs-rest linear model | per-label and macro F1 |
| locate names, products, or symptoms | sequence labeling or span extraction | typed spans | dictionary/rules or CRF | exact span F1 |
| compare a claim with evidence | NLI | relation label | lexical overlap plus rules | relation accuracy and challenge sets |
| find useful documents | retrieval and ranking | ranked list | BM25 | recall@k and nDCG |
| answer from a supplied passage | extractive QA | answer span | sentence retrieval plus matching | exact match and token F1 |
| translate content | conditional generation | target text | phrase-based or small pretrained MT | adequacy, fluency, terminology |
| condense a document | extractive or abstractive summarization | summary | sentence selection | coverage, relevance, faithfulness |
| complete a multi-turn goal | dialogue policy plus generation | action and response | finite-state workflow | task success and safety |
| understand reactions to aspects | aspect sentiment or emotion recognition | aspects and affect labels | lexicon plus linear model | span, label, calibration, and slices |

#### **End-to-End Application Case Study** {#end-to-end-application-case-study}

Consider a multilingual customer-feedback platform. The product requirement is not merely `analyze feedback`; it is to help teams identify urgent problems, understand affected products, find supporting policy, and respond appropriately.

One decomposed design is:

```text
incoming message
-> language and domain detection
-> intent and urgency classification
-> product, order, and issue span extraction
-> aspect-level sentiment and emotion signals
-> policy retrieval
-> grounded answer generation
-> human review for low-confidence or high-impact cases
```

Each output has a different target and metric:

| Component | Example Output | Failure That Matters |
|---|---|---|
| intent classifier | `duplicate_charge` | case routed to the wrong team |
| entity extractor | order `A-1042` | action applied to the wrong account |
| affect model | frustration probability `0.71` | neutral wording treated as hostility |
| retriever | refund policy version 3 | obsolete policy retrieved |
| generator | cited explanation | unsupported promise made to user |
| review policy | escalate to billing specialist | high-risk case handled automatically |

The case study illustrates why task boundaries remain useful even when one large model can technically produce every field. Explicit contracts allow component tests, targeted data, access control, interpretable fallbacks, and failure attribution. A system may still share one encoder or use one generative model, but its behavior should be evaluated against the distinct tasks it claims to perform.

The chapter can be summarized with four questions:

1. What exact information must the system produce?
2. At what granularity should that information be represented?
3. What evidence and supervision justify the target?
4. How will the output be evaluated and used downstream?

Answering these questions turns an interesting language problem into a defensible NLP task. The next chapters in the learning path then provide the tools to compare systems, understand their risks, and integrate them into complete modern NLP applications.